In [0]:
dbutils.widgets.removeAll()

In [0]:
%sql
create widget text storageName default ".";
create widget text catalogwk default "catalog_prod";

In [0]:
%python
storageName = dbutils.widgets.get("storageName")

In [0]:
%sql
CREATE CATALOG IF NOT EXISTS $catalogwk
COMMENT 'Catalogo para la arquitectura medallion del ambiente de dev';

In [0]:
%sql
DROP SCHEMA IF EXISTS catalog_au.raw;
DROP SCHEMA IF EXISTS catalog_au.bronze;
DROP SCHEMA IF EXISTS catalog_au.silver;
DROP SCHEMA IF EXISTS catalog_au.golden;

In [0]:
%python
dbutils.fs.rm(f"s3://s3smartdata2109/bronze",True)
dbutils.fs.rm(f"s3://s3smartdata2109@silver",True)
dbutils.fs.rm(f"s3://s3smartdata2109@golden",True)

In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS catalog_au.raw;
CREATE SCHEMA IF NOT EXISTS catalog_au.bronze;
CREATE SCHEMA IF NOT EXISTS catalog_au.silver;
CREATE SCHEMA IF NOT EXISTS catalog_au.golden;

###Tablas Bronze

In [0]:
%sql
CREATE TABLE IF NOT EXISTS $catalogwk.bronze.circuits (
circuit_id integer,
circuit_ref string,
name string,
location string,
country string,
latitude double,
longitude double,
altitude integer,
ingestion_date timestamp
)
USING DELTA
LOCATION "s3://${storageName}/bronze/circuits"

In [0]:
%sql
CREATE TABLE IF NOT EXISTS $catalogwk.bronze.races (
race_id integer,
race_year integer,
round integer,
circuit_id integer,
name string,
ingestion_date timestamp
)
USING DELTA
PARTITIONED BY (race_year)
LOCATION "s3://${storageName}/bronze/races"

In [0]:
%sql
CREATE TABLE IF NOT EXISTS $catalogwk.bronze.constructors (
  constructor_id integer,
  constructor_ref string,
  name string,
  nationality string,
  ingestion_date timestamp
)
USING DELTA
LOCATION "s3://${storageName}/bronze/constructors"

###Tablas Silver

In [0]:
%sql
CREATE TABLE IF NOT EXISTS $catalogwk.silver.circuits_transformed (
  race_id integer,
  race_year integer,
  round integer,
  circuit_id integer,
  name_race string,
  ingestion_date timestamp,
  circuit_ref string,
  name string,
  location string,
  country string,
  latitude double,
  longitude double,
  altitude integer,
  altitude_category string,
  years_diferences integer,
  lat_diff integer,
  race_type string,
  near_equator string
)
USING DELTA
LOCATION "s3://${storageName}/silver/circuits_transformed"

###Tablas Golden

In [0]:
%sql
CREATE TABLE IF NOT EXISTS $catalogwk.golden.golden_raced_partitioned (
  race_year integer,
  conteo long,
  max_altitude integer,
  min_altitude integer,
  country string,
  race_type string,
  near_equator string
)
USING DELTA
LOCATION "s3://${storageName}/golden/golden_raced_partitioned"